In [1]:
import json
import random
import torch
import numpy as np
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from huggingface_hub import login
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import os

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEEDS = [42, 123, 456]


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class ModelEvaluator:
    def __init__(self, base_model_name: str, trained_model_path: str = None):
        try:
            from google.colab import userdata
            HF_TOKEN = "HF_TOKEN"
            login(token=HF_TOKEN)
        except:
            pass

        print(f"Loading model: {base_model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True
        )

        if trained_model_path:
            print(f"Loading trained model from: {trained_model_path}")
            self.model = PeftModel.from_pretrained(
                base_model,
                trained_model_path,
                is_trainable=False
            )
        else:
            self.model = base_model

        print("Model loaded\n")

    def generate_text(self, prompt: str, max_new_tokens: int = 100) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def generate_from_prompts(self, prompts: list) -> list:
        print(f"Generating from {len(prompts)} prompts...")
        generated = []
        for i, prompt in enumerate(prompts):
            text = self.generate_text(prompt)
            generated.append(text)
            if (i + 1) % 10 == 0:
                print(f"  Generated {i + 1}/{len(prompts)}")
        return generated

    def calculate_lexical_diversity(self, texts: list) -> dict:
        all_tokens = []
        for text in texts:
            tokens = text.lower().split()
            all_tokens.extend(tokens)

        types = len(set(all_tokens))
        tokens_count = len(all_tokens)
        ttr = types / tokens_count if tokens_count > 0 else 0

        bigrams = []
        for text in texts:
            words = text.lower().split()
            bigrams.extend([f"{words[i]}_{words[i+1]}" for i in range(len(words)-1)])
        unique_bigrams = len(set(bigrams))
        total_bigrams = len(bigrams)

        def distinct_n(n):
            all_ngrams = []
            for text in texts:
                words = text.lower().split()
                all_ngrams.extend(zip(*[words[i:] for i in range(n)]))
            return round(len(set(all_ngrams)) / len(all_ngrams), 4) if all_ngrams else 0

        tokenized = [t.lower().split() for t in texts]
        smoother = SmoothingFunction().method1
        self_bleu_scores = []
        sample = tokenized[:min(50, len(tokenized))]
        for i, hyp in enumerate(sample):
            refs = sample[:i] + sample[i+1:]
            if refs:
                self_bleu_scores.append(sentence_bleu(refs, hyp, smoothing_function=smoother))
        self_bleu_score = round(float(np.mean(self_bleu_scores)), 4) if self_bleu_scores else 0

        return {
            'type_token_ratio': round(ttr, 4),
            'vocabulary_size': types,
            'total_tokens': tokens_count,
            'bigram_diversity': round(unique_bigrams / total_bigrams if total_bigrams > 0 else 0, 4),
            'distinct_1': distinct_n(1),
            'distinct_2': distinct_n(2),
            'distinct_3': distinct_n(3),
            'self_bleu': self_bleu_score
        }

    def calculate_perplexity(self, texts: list) -> float:
        total_loss = 0
        count = 0

        for text in texts:
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                total_loss += outputs.loss.item()
                count += 1

        avg_loss = total_loss / count
        perplexity = np.exp(avg_loss)
        return round(float(perplexity), 2)


def calculate_factual_consistency(generated_texts: list, reference_texts: list, device: int = 0) -> float:
    """NLI-based factual consistency: mean entailment probability of generated sentences vs. reference prompt."""
    nli = pipeline(
        "text-classification",
        model="cross-encoder/nli-deberta-v3-small",
        device=device,
        top_k=None
    )
    scores = []
    for gen, ref in zip(generated_texts, reference_texts):
        sentences = nltk.sent_tokenize(gen)
        for sent in sentences:
            if len(sent.strip()) < 10:
                continue
            pair = f"{ref} [SEP] {sent}"
            result = nli(pair[:512], truncation=True)
            if isinstance(result[0], list):  # top_k=None wraps output in extra list
                result = result[0]
            for r in result:
                if r['label'].upper() == 'ENTAILMENT':
                    scores.append(r['score'])
                    break
    return round(float(np.mean(scores)), 4) if scores else 0.0


def load_test_prompts(filepath: str, n: int = 50):
    with open(filepath, 'r') as f:
        data = [json.loads(line) for line in f]

    prompts = []
    for item in data[:n*2]:
        words = item['text'].split()
        if len(words) >= 20:
            prompt = ' '.join(words[:15])
            prompts.append(prompt)
            if len(prompts) >= n:
                break
    return prompts


def run_single_seed_evaluation(evaluator: ModelEvaluator, prompts: list) -> dict:
    generated = evaluator.generate_from_prompts(prompts)
    lexical = evaluator.calculate_lexical_diversity(generated)
    perplexity = evaluator.calculate_perplexity(generated)
    factual = calculate_factual_consistency(generated, prompts)
    return {
        'lexical_diversity': lexical,
        'perplexity': perplexity,
        'factual_consistency': factual,
        'num_prompts': len(prompts),
        'generated_texts': generated
    }


def aggregate_seed_results(seed_results: list) -> dict:
    """Compute mean ± std across seed runs."""
    metric_keys = ['type_token_ratio', 'vocabulary_size', 'bigram_diversity',
                   'distinct_1', 'distinct_2', 'distinct_3', 'self_bleu']
    agg = {'lexical_diversity': {}, 'perplexity': {}, 'factual_consistency': {}}

    for key in metric_keys:
        vals = [r['lexical_diversity'][key] for r in seed_results]
        agg['lexical_diversity'][key] = {'mean': round(np.mean(vals), 4), 'std': round(np.std(vals), 4)}

    ppl_vals = [r['perplexity'] for r in seed_results]
    agg['perplexity'] = {'mean': round(np.mean(ppl_vals), 2), 'std': round(np.std(ppl_vals), 2)}

    fc_vals = [r['factual_consistency'] for r in seed_results]
    agg['factual_consistency'] = {'mean': round(np.mean(fc_vals), 4), 'std': round(np.std(fc_vals), 4)}

    agg['num_seeds'] = len(seed_results)
    return agg


def main():
    from google.colab import drive
    drive.mount('/content/drive')

    project_root = "/content/drive/MyDrive/FinalProject"
    BASE_MODEL = "meta-llama/Llama-3.2-1B"

    test_file = f"{project_root}/human_baseline_data/test.jsonl"
    prompts = load_test_prompts(test_file, n=50)
    print(f"Loaded {len(prompts)} test prompts\n")

    models_config = {
        'Base (Untrained)': None,
        'Human-trained': f"{project_root}/trained_models_v2/human_baseline_data_llama",
        'AI-trained': f"{project_root}/trained_models_v2/ai_generated_data_gpt2_medium_llama",
        'Mixed-trained': f"{project_root}/trained_models_v2/mixed_data_gpt2_medium_llama"
    }

    final_results = {}

    for model_name, model_path in models_config.items():
        print("="*60)
        print(f"EVALUATING: {model_name}")
        print("="*60)

        try:
            evaluator = ModelEvaluator(BASE_MODEL, model_path)
            seed_results = []

            for seed in SEEDS:
                print(f"\n  -- Seed {seed} --")
                set_seed(seed)
                result = run_single_seed_evaluation(evaluator, prompts)
                seed_results.append(result)

            agg = aggregate_seed_results(seed_results)
            final_results[model_name] = agg

            ld = agg['lexical_diversity']
            print(f"\n{model_name} (mean ± std over {len(SEEDS)} seeds):")
            print(f"  TTR:                {ld['type_token_ratio']['mean']:.4f} ± {ld['type_token_ratio']['std']:.4f}")
            print(f"  Vocab Size:         {ld['vocabulary_size']['mean']:.1f} ± {ld['vocabulary_size']['std']:.1f}")
            print(f"  Bigram Diversity:   {ld['bigram_diversity']['mean']:.4f} ± {ld['bigram_diversity']['std']:.4f}")
            print(f"  Distinct-1:         {ld['distinct_1']['mean']:.4f} ± {ld['distinct_1']['std']:.4f}")
            print(f"  Distinct-2:         {ld['distinct_2']['mean']:.4f} ± {ld['distinct_2']['std']:.4f}")
            print(f"  Distinct-3:         {ld['distinct_3']['mean']:.4f} ± {ld['distinct_3']['std']:.4f}")
            print(f"  Self-BLEU:          {ld['self_bleu']['mean']:.4f} ± {ld['self_bleu']['std']:.4f}")
            print(f"  Perplexity:         {agg['perplexity']['mean']:.2f} ± {agg['perplexity']['std']:.2f}")
            print(f"  Factual Consistency:{agg['factual_consistency']['mean']:.4f} ± {agg['factual_consistency']['std']:.4f}")

            del evaluator
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"Error: {e}")
            import traceback
            traceback.print_exc()
            print("Skipping...\n")
            continue

    # Summary table
    print("\n" + "="*80)
    print(f"FINAL EVALUATION RESULTS ({len(prompts)} prompts, {len(SEEDS)} seeds each)")
    print("="*80)
    print(f"\n{'Model':<20} {'TTR':<14} {'Distinct-2':<14} {'Self-BLEU':<14} {'Factual':<14} {'PPL':<10}")
    print("-" * 90)
    for model_name, metrics in final_results.items():
        ld = metrics['lexical_diversity']
        ttr = f"{ld['type_token_ratio']['mean']:.4f}±{ld['type_token_ratio']['std']:.4f}"
        d2  = f"{ld['distinct_2']['mean']:.4f}±{ld['distinct_2']['std']:.4f}"
        sb  = f"{ld['self_bleu']['mean']:.4f}±{ld['self_bleu']['std']:.4f}"
        fc  = f"{metrics['factual_consistency']['mean']:.4f}±{metrics['factual_consistency']['std']:.4f}"
        ppl = f"{metrics['perplexity']['mean']:.2f}±{metrics['perplexity']['std']:.2f}"
        print(f"{model_name:<20} {ttr:<14} {d2:<14} {sb:<14} {fc:<14} {ppl:<10}")

    # Save
    results_path = f"{project_root}/evaluation_results_final.json"
    with open(results_path, 'w') as f:
        json.dump(final_results, f, indent=2)

    print(f"\n✓ Results saved to: {results_path}")
    print("="*80)
    print("EVALUATION COMPLETE")
    print("="*80)

    return final_results


if __name__ == "__main__":
    results = main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 50 test prompts

EVALUATING: Base (Untrained)
Loading model: meta-llama/Llama-3.2-1B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded


  -- Seed 42 --
Generating from 50 prompts...
  Generated 10/50
  Generated 20/50
  Generated 30/50
  Generated 40/50
  Generated 50/50


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  -- Seed 456 --
Generating from 50 prompts...
  Generated 10/50
  Generated 20/50
  Generated 30/50
  Generated 40/50
  Generated 50/50


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Mixed-trained (mean ± std over 3 seeds):
  TTR:                0.3526 ± 0.0062
  Vocab Size:         1628.0 ± 21.2
  Bigram Diversity:   0.7657 ± 0.0154
  Distinct-1:         0.3526 ± 0.0062
  Distinct-2:         0.7657 ± 0.0154
  Distinct-3:         0.9250 ± 0.0174
  Self-BLEU:          0.0560 ± 0.0030
  Perplexity:         5.67 ± 0.21
  Factual Consistency:0.0506 ± 0.0059

FINAL EVALUATION RESULTS (50 prompts, 3 seeds each)

Model                TTR            Distinct-2     Self-BLEU      Factual        PPL       
------------------------------------------------------------------------------------------
Base (Untrained)     0.3759±0.0043  0.7717±0.0128  0.0498±0.0038  0.0544±0.0068  6.32±0.09 
Human-trained        0.3652±0.0079  0.7909±0.0127  0.0579±0.0039  0.0503±0.0076  7.82±0.02 
AI-trained           0.2766±0.0070  0.5896±0.0239  0.0697±0.0033  0.0793±0.0097  3.73±0.16 
Mixed-trained        0.3526±0.0062  0.7657±0.0154  0.0560±0.0030  0.0506±0.0059  5.67±0.21 

✓ Results saved 